In [13]:
!pip install evaluate
!pip install seqeval

import json
from collections import Counter
import numpy as np
import evaluate
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification, 
    TrainingArguments, 
    Trainer, 
    TrainerCallback,
    DataCollatorForTokenClassification
)
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score
import shutil
import os
import pandas as pd
from IPython.display import display
import torch
import torch.nn as nn

# 1️⃣ CONFIGURATION & PATHS
# Points directly to your processed files in Kaggle working directory
TRAIN_PATH = "/kaggle/input/datasets/abdullahshheikh/pii-masking-augmented/train_processed.json"
TEST_PATH = "/kaggle/input/datasets/abdullahshheikh/pii-masking-augmented/test_processed.json"
MODEL_CHECKPOINT = "microsoft/deberta-v3-small"

# Label mapping - Ensure this matches your JSON tags exactly
label_list = ["O", "B-PER", "I-PER", "B-EMAIL", "I-EMAIL"]
label_to_id = {label: i for i, label in enumerate(label_list)}
id_to_label = {i: label for i, label in enumerate(label_list)}

# 2️⃣ DATA LOADING
with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_data = json.load(f)
with open(TEST_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

# Convert to Dataset objects
full_train_ds = Dataset.from_list(train_data)

# Split once into train and validation
split_ds = full_train_ds.train_test_split(test_size=0.1, seed=42)

raw_datasets = DatasetDict({
    "train": split_ds["train"],             # 90% of training data
    "validation": split_ds["test"],         # 10% of training data
    "test": Dataset.from_list(test_data)    # Completely unseen test set
})

# Check if labels actually exist in the raw data
all_labels = [tag for entry in train_data for tag in entry["ner_tags"]]
unique, counts = np.unique(all_labels, return_counts=True)
label_counts = dict(zip(unique, counts))

print("--- RAW DATA CHECK ---")
print(f"Total tokens: {len(all_labels)}")
print(f"Label distribution: {label_counts}")

if "B-PER" not in label_counts and "B-EMAIL" not in label_counts:
    print("❌ ERROR: No PII found in raw data. Check your JSON keys.")


--- RAW DATA CHECK ---
Total tokens: 731505
Label distribution: {np.str_('B-EMAIL'): np.int64(11416), np.str_('B-PER'): np.int64(40264), np.str_('I-PER'): np.int64(29466), np.str_('O'): np.int64(650359)}


In [14]:
# 3️⃣ TOKENIZATION & ALIGNMENT
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def align_labels(examples):
    tokenized = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        
        for word_idx in word_ids:
            if word_idx is None:
                # Special tokens like [CLS], [SEP]
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # This is the FIRST sub-token of a new word
                label_ids.append(label_to_id[label[word_idx]])
            else:
                # This is a CONTINUATION sub-token of the same word
                full_label = label[word_idx]
                if full_label.startswith("B-"):
                    # Change B-PER to I-PER for sub-tokens
                    new_label = "I-" + full_label[2:]
                    label_ids.append(label_to_id[new_label])
                else:
                    # Keep I- or O labels as they are
                    label_ids.append(label_to_id[full_label])
            
            previous_word_idx = word_idx
        labels.append(label_ids)
    
    tokenized["labels"] = labels
    return tokenized
tokenized_ds = raw_datasets.map(align_labels, batched=True)


Map:   0%|          | 0/25664 [00:00<?, ? examples/s]

Map:   0%|          | 0/2852 [00:00<?, ? examples/s]

Map:   0%|          | 0/3650 [00:00<?, ? examples/s]

In [15]:
def count_token_labels(dataset, label_map):
    # Flatten all labels, excluding the -100 (ignore) index
    all_labels = [
        label_map[l] 
        for seq in dataset["labels"] 
        for l in seq 
        if l != -100
    ]
    
    counts = Counter(all_labels)
    total = sum(counts.values())
    
    print(f"{'Tag':<10} | {'Count':<10} | {'Percentage':<10}")
    print("-" * 35)
    for tag in ["O", "B-PER", "I-PER", "B-EMAIL", "I-EMAIL"]:
        count = counts.get(tag, 0)
        percentage = (count / total) * 100 if total > 0 else 0
        print(f"{tag:<10} | {count:<10} | {percentage:>8.4f}%")

# Check the training and validation splits
print("--- TRAINING TAG DISTRIBUTION ---")
count_token_labels(tokenized_ds["train"], id_to_label)

print("\n--- VALIDATION TAG DISTRIBUTION ---")
count_token_labels(tokenized_ds["validation"], id_to_label)

--- TRAINING TAG DISTRIBUTION ---
Tag        | Count      | Percentage
-----------------------------------
O          | 620610     |  77.7967%
B-PER      | 36286      |   4.5486%
I-PER      | 53797      |   6.7437%
B-EMAIL    | 10243      |   1.2840%
I-EMAIL    | 76797      |   9.6269%

--- VALIDATION TAG DISTRIBUTION ---
Tag        | Count      | Percentage
-----------------------------------
O          | 69358      |  77.9654%
B-PER      | 3978       |   4.4717%
I-PER      | 5689       |   6.3950%
B-EMAIL    | 1173       |   1.3186%
I-EMAIL    | 8762       |   9.8494%


In [16]:
# 4️⃣ METRICS (Precision, Recall, F1, Accuracy, FPR, FNR)
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index -100 and convert IDs to labels
    true_preds = [[id_to_label[p] for (p, l) in zip(pred, lab) if l != -100] for pred, lab in zip(predictions, labels)]
    true_labs = [[id_to_label[l] for (p, l) in zip(pred, lab) if l != -100] for pred, lab in zip(predictions, labels)]

    # Flatten lists for scikit-learn metrics
    flat_preds = [t for seq in true_preds for t in seq]
    flat_labs = [t for seq in true_labs for t in seq]

    final_metrics = {}

    # Calculate metrics for PER and EMAIL specifically
    for tag in ["PER", "EMAIL"]:
        # Binary classification: Is it this tag (B or I) or not?
        y_true_binary = [1 if tag in t else 0 for t in flat_labs]
        y_pred_binary = [1 if tag in t else 0 for t in flat_preds]
        
        tn, fp, fn, tp = confusion_matrix(y_true_binary, y_pred_binary, labels=[0,1]).ravel()
        
        # Flattened keys so Trainer can log them
        final_metrics[f"{tag}_precision"] = precision_score(y_true_binary, y_pred_binary, zero_division=0)
        final_metrics[f"{tag}_recall"] = recall_score(y_true_binary, y_pred_binary, zero_division=0)
        final_metrics[f"{tag}_f1"] = f1_score(y_true_binary, y_pred_binary, zero_division=0)
        final_metrics[f"{tag}_FPR"] = fp / (fp + tn) if (fp + tn) > 0 else 0
        final_metrics[f"{tag}_FNR"] = fn / (fn + tp) if (fn + tp) > 0 else 0

    # Add overall seqeval results (standard NER metrics)
    seq_results = seqeval.compute(predictions=true_preds, references=true_labs)
    final_metrics["overall_precision"] = seq_results["overall_precision"]
    final_metrics["overall_recall"] = seq_results["overall_recall"]
    final_metrics["overall_f1"] = seq_results["overall_f1"]
    final_metrics["overall_accuracy"] = seq_results["overall_accuracy"]

    return final_metrics

class NERDebugCallback(TrainerCallback):
    def __init__(self, debug_tokenizer, debug_model, debug_dataset):
        self.debug_tokenizer = debug_tokenizer
        self.debug_model = debug_model
        self.debug_dataset = debug_dataset

    def on_evaluate(self, args, state, control, **kwargs):
        print(f"\n--- 🧪 DEBUG: PREDICTION CHECK (Step {state.global_step}) ---")

        self.debug_model.eval()
        
        # Grab the first sample directly from the dataset we passed in
        sample = self.debug_dataset[0]
        
        # Prepare inputs
        inputs = {
            "input_ids": torch.tensor(sample["input_ids"]).unsqueeze(0).to(self.debug_model.device),
            "attention_mask": torch.tensor(sample["attention_mask"]).unsqueeze(0).to(self.debug_model.device)
        }

        with torch.no_grad():
            outputs = self.debug_model(**inputs)
            # Use softmax to see how much the model "considers" PII
            probs = torch.nn.functional.softmax(outputs.logits.float(), dim=-1)
            preds = torch.argmax(outputs.logits, dim=-1).squeeze().tolist()
            labels = sample["labels"]
            tokens = self.debug_tokenizer.convert_ids_to_tokens(sample["input_ids"])

        print(f"{'Token':<15} | {'True':<12} | {'Pred':<12} | {'PII Prob (%)':<12}")
        print("-" * 60)

        found_something = False
        for i, (t, l, p) in enumerate(zip(tokens, labels, preds)):
            if l != -100:
                # Sum of probabilities for all PII tags (labels 1-4)
                pii_prob = torch.sum(probs[0][i][1:]).item() * 100
                
                # Print if it's actually PII or if the model thinks it might be (>1% confidence)
                if l > 0 or p > 0 or pii_prob > 1.0:
                    print(f"{t:<15} | {id_to_label[l]:<12} | {id_to_label[p]:<12} | {pii_prob:.2f}%")
                    found_something = True
        
        if not found_something:
            print("Model is 99%+ confident everything in this sample is 'O'.")

        self.debug_model.train()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_weights = torch.tensor([1.0, 3.0, 3.0, 4.0, 3.0]).to(device)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        current_step = self.state.global_step
        unique_ids = torch.unique(labels).cpu().tolist()
        # Filter out -100 to see actual class labels (0, 1, 2, 3, 4)
        actual_labels = [l for l in unique_ids if l != -100]
        has_pii = any(l > 0 for l in actual_labels)

        # Print every 10 steps so we can see the data distribution
        if current_step % 10 == 0:
            print(f"📡 [STEP {current_step}] Labels in Batch: {actual_labels} | PII Present: {has_pii}")
            
        outputs = model(**inputs)
        logits = outputs.get("logits").float()
        
        # DEBUG 1: Check if the model is already broken
        if torch.isnan(logits).any():
            print("🚨 CRITICAL: Logits are NaN before loss calculation!")

        current_weights = class_weights.to(device=logits.device, dtype=logits.dtype)
        loss_fct = nn.CrossEntropyLoss(weight=current_weights, ignore_index=-100)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        
        # DEBUG 2: Check if the weights caused the loss to explode
        if torch.isnan(loss):
            print(f"🚨 CRITICAL: Loss became NaN! Max Logit: {logits.max().item()}")

        if torch.isnan(logits).any():
            print("NaN detected in logits!")
            print("Max logit:", torch.max(logits))
            print("Min logit:", torch.min(logits))
            print("Batch input ids:", inputs["input_ids"][0][:20])
            raise ValueError("Stopping due to NaN logits")
            
        return (loss, outputs) if return_outputs else loss

# 5️⃣ MODEL TRAINING
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT, 
    num_labels=len(label_list), 
    id2label=id_to_label, 
    label2id=label_to_id
)

# Stability fix for DeBERTa-v3 NaN issues
model.config.layer_norm_eps = 1e-6
model = model.float()
gradient_accumulation_steps=2

args = TrainingArguments(
    output_dir="./pii_deberta_results",
    per_device_train_batch_size=8, 
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    learning_rate=8e-6,                # 👈 Lowered back to safe levels
    weight_decay=0.01,
    warmup_ratio=0.1,                  # 👈 Re-introduced warmup to stabilize start
    lr_scheduler_type="cosine",        
    
    fp16=False,
    bf16=False,
    max_grad_norm=0.3,                 # 👈 Strict gradient clipping to stop NaNs
    
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="overall_f1", 
    logging_steps=10
)

# Initialize the trainer with the new class
trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[NERDebugCallback(
        debug_tokenizer=tokenizer, 
        debug_model=model, 
        debug_dataset=tokenized_ds["validation"]
    )]
)

model.gradient_checkpointing_enable()
model = model.float()

# Run the final check before training
sample = tokenized_ds["train"][0]
print("Tokens:", tokenizer.convert_ids_to_tokens(sample["input_ids"][:20]))
print("Labels:", [id_to_label[l] if l != -100 else -100 for l in sample["labels"][:20]])

# START TRAINING
# Check what the trainer is actually seeing in a batch
train_dataloader = trainer.get_train_dataloader()
batch = next(iter(train_dataloader))
print("Labels in first batch:", batch['labels'][0])

print("Max input id:", batch["input_ids"].max())
print("Vocab size:", model.config.vocab_size)
trainer.train()

# 6️⃣ FINAL EVALUATION
print("\n--- FINAL METRICS ON UNSEEN TEST DATA ---")
print(trainer.evaluate(tokenized_ds["test"]))

# 1. Save the final model and tokenizer in a clean directory
save_directory = "/kaggle/working/final_pii_model"
trainer.save_model(save_directory)
tokenizer.save_pretrained(save_directory)

# 2. Create a ZIP file of that directory
# This creates 'pii_model_download.zip' in your /kaggle/working/ folder
shutil.make_archive("pii_model_download", 'zip', save_directory)

print(f"--- SUCCESS ---")
print(f"Model saved to: {save_directory}")
print(f"ZIP file created: /kaggle/working/pii_model_download.zip")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForTokenClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture;

Tokens: ['▁so', '▁he', '▁was', '▁replaced', '▁with', '▁Al', '▁Corley', '▁,', '▁who', '▁originated', '▁the', '▁part', '▁in', '▁1981', '▁.']
Labels: ['O', 'O', 'O', 'O', 'O', 'B-PER', 'I-PER', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
Labels in first batch: tensor([   0,    0,    0,    0,    0,    0,    0,    1,    0,    3,    4,    4,
           4,    4,    4,    4,    4,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100], device='cuda:0')
Max input id: tensor(101360, device='cuda:0')
Vocab size: 128100
📡 [STEP 0] Labels in Batch: [0, 1, 2, 3, 4] | PII Present: True
📡 [STEP 0] Labels in Batch: [0, 1, 2, 3, 4] | PII Present: True


Epoch,Training Loss,Validation Loss,Per Precision,Per Recall,Per F1,Per Fpr,Per Fnr,Email Precision,Email Recall,Email F1,Email Fpr,Email Fnr,Overall Precision,Overall Recall,Overall F1,Overall Accuracy
1,0.040849,0.013948,0.978560,0.991518,0.984996,0.002648,0.008482,0.999799,1.000000,0.999899,0.000025,0.000000,0.971009,0.988352,0.979604,0.996583
2,0.012998,0.013361,0.983477,0.991311,0.987378,0.002030,0.008689,1.000000,1.000000,1.000000,0.000000,0.000000,0.975129,0.989517,0.982270,0.997145
3,0.024620,0.013113,0.978513,0.994000,0.986196,0.002661,0.006000,1.000000,1.000000,1.000000,0.000000,0.000000,0.973323,0.991652,0.982402,0.996864
4,0.012081,0.012927,0.985300,0.991518,0.988399,0.001803,0.008482,1.000000,1.000000,1.000000,0.000000,0.000000,0.980762,0.989711,0.985216,0.997347
5,0.002247,0.013505,0.985811,0.991828,0.988810,0.001740,0.008172,1.000000,1.000000,1.000000,0.000000,0.000000,0.980573,0.989711,0.985121,0.997437


📡 [STEP 10] Labels in Batch: [0, 1, 2, 3, 4] | PII Present: True
📡 [STEP 10] Labels in Batch: [0, 1, 2, 3, 4] | PII Present: True
📡 [STEP 20] Labels in Batch: [0, 1, 2, 3, 4] | PII Present: True
📡 [STEP 20] Labels in Batch: [0, 1, 2, 3, 4] | PII Present: True
📡 [STEP 30] Labels in Batch: [0, 1, 2, 3, 4] | PII Present: True
📡 [STEP 30] Labels in Batch: [0, 1, 2, 3, 4] | PII Present: True
📡 [STEP 40] Labels in Batch: [0, 1, 2, 3, 4] | PII Present: True
📡 [STEP 40] Labels in Batch: [0, 1, 2, 3, 4] | PII Present: True
📡 [STEP 50] Labels in Batch: [0, 1, 2, 3, 4] | PII Present: True
📡 [STEP 50] Labels in Batch: [0, 1, 2, 3, 4] | PII Present: True
📡 [STEP 60] Labels in Batch: [0, 1, 2, 3, 4] | PII Present: True
📡 [STEP 60] Labels in Batch: [0, 1, 2, 3, 4] | PII Present: True
📡 [STEP 70] Labels in Batch: [0, 1, 2, 3, 4] | PII Present: True
📡 [STEP 70] Labels in Batch: [0, 1, 2, 3, 4] | PII Present: True
📡 [STEP 80] Labels in Batch: [0, 1, 2, 3, 4] | PII Present: True
📡 [STEP 80] Labels in Bat

...

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye


--- FINAL METRICS ON UNSEEN TEST DATA ---
📡 [STEP 8020] Labels in Batch: [0, 1, 2, 3, 4] | PII Present: True


📡 [STEP 8020] Labels in Batch: [0, 1, 2, 3, 4] | PII Present: True
📡 [STEP 8020] Labels in Batch: [0, 1, 2, 3, 4] | PII Present: True
📡 [STEP 8020] Labels in Batch: [0, 1, 2, 3, 4] | PII Present: True
📡 [STEP 8020] Labels in Batch: [0, 1, 2, 3, 4] | PII Present: True
📡 [STEP 8020] Labels in Batch: [0, 1, 2, 3, 4] | PII Present: True
📡 [STEP 8020] Labels in Batch: [0, 1, 2, 3, 4] | PII Present: True
📡 [STEP 8020] Labels in Batch: [0, 1, 2, 3, 4] | PII Present: True
📡 [STEP 8020] Labels in Batch: [0, 1, 2, 3, 4] | PII Present: True
📡 [STEP 8020] Labels in Batch: [0, 1, 2] | PII Present: True
📡 [STEP 8020] Labels in Batch: [0, 1, 2, 3, 4] | PII Present: True
📡 [STEP 8020] Labels in Batch: [0, 1, 2, 3, 4] | PII Present: True
📡 [STEP 8020] Labels in Batch: [0, 1, 2, 3, 4] | PII Present: True
📡 [STEP 8020] Labels in Batch: [0, 1, 2] | PII Present: True
📡 [STEP 8020] Labels in Batch: [0, 1, 2, 3, 4] | PII Present: True
📡 [STEP 8020] Labels in Batch: [0, 1, 2, 3, 4] | PII Present: True
📡 [STEP

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

--- SUCCESS ---
Model saved to: /kaggle/working/final_pii_model
ZIP file created: /kaggle/working/pii_model_download.zip
